# 66. 비라벨 공정 상태 군집과 누락 가스 분포 — 탐색 전용
MAZIN의양품중심거리접근과공정군집연구를참고한다. 6개관계피처로비라벨공정군집8개를만들고정답을나중에대조한다. 비라벨을정상이라고가정하지않는다. 여기서군집은공정상태일뿐가스원인정답아니다. 전체개발자료탐색이므로이군집을그대로평가모델에재사용하면안된다. 모델평가에서는각외부학습구간안에서비라벨정규화·군집을다시학습해야한다.
RPM은학습/추가기간간정의문제때문에이군집입력에서제외한다. 이미개봉한37개도이제개발자료이며독립테스트로재사용하지않는다.


In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
cols=['EQUIP_CD','TimeStamp','PART_NAME','PassOrFail','Reason','Barrel_Temperature_2','Barrel_Temperature_5','Mold_Temperature_3','Max_Injection_Pressure','Max_Switch_Over_Pressure','Cushion_Position','Plasticizing_Position','Average_Back_Pressure','Cycle_Time','Plasticizing_Time']
l=pd.read_csv(root/'data/original/labeled_data.csv',usecols=cols)
l=l[l.PART_NAME.str.contains('RG3',na=False)&l.PART_NAME.str.contains('RH',na=False)].copy()
l.TimeStamp=pd.to_datetime(l.TimeStamp)
l['gas']=l.PassOrFail.astype(str).str.upper().eq('N')&l.Reason.fillna('').str.contains('가스')
l=l[l.gas|l.PassOrFail.astype(str).str.upper().eq('Y')].reset_index(drop=True)
u=pd.read_csv(root/'data/original_split/moldset_unlabeled_rg3.csv')
u=u[u.PART_NAME.str.contains('RH',na=False)].copy();u.TimeStamp=pd.to_datetime(u.TimeStamp)
keys=['EQUIP_CD','TimeStamp']
u=u.drop_duplicates(keys)
seen=pd.MultiIndex.from_frame(l[keys])
u=u[~pd.MultiIndex.from_frame(u[keys]).isin(seen)]
u=u[(u.Barrel_Temperature_5>0)&(u.Max_Injection_Pressure>0)&(u.Plasticizing_Position>0)]
def features(x):
 return pd.DataFrame({
 'barrel2_minus5':x.Barrel_Temperature_2-x.Barrel_Temperature_5,
 'barrel5_mold_gap':x.Barrel_Temperature_5-x.Mold_Temperature_3,
 'pressure_switch_gap':x.Max_Injection_Pressure-x.Max_Switch_Over_Pressure,
 'cushion_meter_ratio':x.Cushion_Position/x.Plasticizing_Position.where(x.Plasticizing_Position.abs()>1e-6),
 'back_pressure':x.Average_Back_Pressure,
 'cycle_minus_recovery':x.Cycle_Time-x.Plasticizing_Time}).replace([np.inf,-np.inf],np.nan)
imp=SimpleImputer(strategy='median').fit(features(u));sc=RobustScaler().fit(imp.transform(features(u)))
zu=sc.transform(imp.transform(features(u)));zl=sc.transform(imp.transform(features(l)))
cl=MiniBatchKMeans(n_clusters=8,random_state=42,n_init=10,batch_size=512).fit(zu)
l['regime_cluster']=cl.predict(zl)
old=pd.read_csv(tables/'rg3_pdf_fixed_majority_predictions.csv',parse_dates=['TimeStamp'])
old=old[old.scheme.eq('leave_one_block_out')][['TimeStamp','prediction']]
l=l.merge(old,on='TimeStamp',how='left',validate='one_to_one')
l['previously_missed']=l.gas&l.prediction.eq(0)
l.to_csv(tables/'rg3_regime_cluster_development_cases.csv',index=False,encoding='utf-8-sig')
summary=l.groupby('regime_cluster').agg(labeled=('TimeStamp','size'),gas=('gas','sum'),prior_misses=('previously_missed','sum'),start=('TimeStamp','min'),end=('TimeStamp','max'))
summary.to_csv(tables/'rg3_regime_cluster_development_summary.csv',encoding='utf-8-sig')
pd.DataFrame(sc.inverse_transform(cl.cluster_centers_),columns=features(u).columns).to_csv(tables/'rg3_regime_cluster_centers.csv',index_label='cluster',encoding='utf-8-sig')
print('Unlabeled valid unique RH records',len(u),'labeled development',len(l),'gas',l.gas.sum())
print(summary.to_string())
print('Missed gas cluster assignments')
print(l[l.previously_missed][['TimeStamp','regime_cluster']].to_string(index=False))
print('EXPLORATORY ONLY. Full-development clustering is not an independent performance test.')
